In [ ]:
# ============================================================
# NARX (MLP) con LAGS=6 para dataset V4 (con Provedores/Unidad)
#
# OBJETIVO:
#   Predecir la siguiente semana por producto usando:
#     1) Últimas 6 semanas del producto  (autoregresivo)
#     2) Últimas 6 semanas del TOTAL semanal (exógena)
#
# EXTRA:
#   Mostrar los grupos (proveedores) y dejar que el usuario elija:
#     - 0 = ALL (todos)
#     - 1..N = un grupo específico
# ============================================================

# ============================================================
# A) AQUÍ IMPORTAMOS TODO LO QUE VAMOS A USAR
# ============================================================
import pandas as pd
import numpy as np
import torch #biblioteca principal de PyTorch, que es un framework de inteligencia artificial y redes neuronales.
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import re #es el módulo de expresiones regulares (Regular Expressions)


# ============================================================
# B) AQUÍ CONFIGURAMOS TODO (ARCHIVO, LAGS, EPOCHS, ETC.)
# ============================================================
CSV_PATH  = "Panquel_Dataset_V5_Provedores.csv"  # <-- aquí pones tu dataset V4
#CSV_PATH  = "Panquel_Dataset_V5_Enero_Pruebas.csv"   # <-- aquí pruebas
LAGS      = 6                                          # <-- aquí fijas que usaremos 6 semanas
TEST_SIZE = 0.2
EPOCHS    = 400
LR        = 0.01
SEED      = 42

# Aquí fijamos semilla para que el resultado sea lo más reproducible posible
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# C) AQUÍ SE LEE EL CSV Y SE SEPARA:
#    - columnas de "meta" (ID, Producto, etc.)
#    - columnas de semanas (lo numérico)
# ============================================================
df = pd.read_csv(CSV_PATH)

META_COLS = ["ID", "Producto", "Unidad", "Provedores"]  # <-- columnas NO temporales
PRODUCT_COL = "Producto"                                # <-- nombre del producto
GROUP_COL = "Provedores"                                # <-- aquí están los "grupos"

# Aquí validamos que el CSV tenga las columnas necesarias
if PRODUCT_COL not in df.columns or GROUP_COL not in df.columns:
    raise ValueError(f"Faltan columnas requeridas. Columnas disponibles: {list(df.columns)}")

# Aquí detectamos automáticamente cuáles columnas son semanas
week_cols = [c for c in df.columns if c not in META_COLS]

# Aquí convertimos las semanas a números (por si hay textos/vacíos)
weeks_df = df[week_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

# Aquí dejamos el dataset en forma matriz:
#    data = (productos, semanas)
data = weeks_df.values.astype(np.float32)

n_products, n_weeks = data.shape
print(f" Dataset cargado: {n_products} productos | {n_weeks} semanas")
print(f" Semanas detectadas: {week_cols[0]} ... {week_cols[-1]}")


# ============================================================
# D) AQUÍ SE CREA LA EXÓGENA:
#    "TOTAL semanal" = suma de todos los productos en cada semana
# ============================================================
# Esto funciona como variable externa (X) para NARX
exo_total = data.sum(axis=0).astype(np.float32)
print(" Exógena creada: total semanal (global)")


# ============================================================
# E) ESTA FUNCIÓN SIRVE PARA CREAR LAS VENTANAS NARX
#    (o sea, convertir series de tiempo en datos supervisados)
# ============================================================
def make_windows_narx(data_matrix, exo_series, lags=6):
    """
    Esta función "prepara" los datos para entrenar.

    Para cada producto p y para cada semana t:
      - Entrada X:
          y_lags = y(t-6..t-1)  (historial del producto)
          x_lags = x(t-6..t-1)  (historial del total semanal)
          X = concat(y_lags, x_lags)  -> tamaño = 12

      - Salida y:
          y(t)  (valor real de la semana t)

    Retorna:
      X -> (muestras, 12)
      y -> (muestras,)
    """
    X_list, y_list = [], []
    n_products, n_weeks = data_matrix.shape

    # Recorremos producto por producto
    for p in range(n_products):
        y_series = data_matrix[p]

        # Recorremos el tiempo a partir de la semana 6 (porque antes no hay lags suficientes)
        for t in range(lags, n_weeks):
            y_lags = y_series[t-lags:t]      # últimas 6 del producto
            x_lags = exo_series[t-lags:t]    # últimas 6 del total semanal

            X_list.append(np.concatenate([y_lags, x_lags]))
            y_list.append(y_series[t])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


# AQUÍ SE CREAN LOS DATOS SUPERVISADOS PARA ENTRENAR
X, y = make_windows_narx(data, exo_total, lags=LAGS)
print(f"Ventanas creadas: X={X.shape} | y={y.shape}")


# ============================================================
# F) AQUÍ SE HACE EL TRAIN/TEST SPLIT SIN MEZCLAR (SERIES DE TIEMPO)
# ============================================================
# IMPORTANTE: shuffle=False porque en series de tiempo no se deben mezclar las semanas
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False
)

# AQUÍ PASAMOS TODO A TENSORES PARA PYTORCH
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train).unsqueeze(1)  # (N,1) porque salida es un valor
X_test_t  = torch.tensor(X_test)
y_test_t  = torch.tensor(y_test).unsqueeze(1)

print(f"Split: Train={X_train.shape} | Test={X_test.shape}")


# ============================================================
# G) AQUÍ DEFINIMOS LA RED NEURONAL (EL MODELO NARX)
# ============================================================
class NeuralNARX(nn.Module):
    """
    Esta clase es la red neuronal.
    Sirve como f(...) en:
        y(t) = f( y_lags, x_lags )
    """
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),  # 12 -> 64
            nn.ReLU(),
            nn.Linear(64, 32),          # 64 -> 32
            nn.ReLU(),
            nn.Linear(32, 1)            # 32 -> 1 (predicción final)
        )

    def forward(self, x):
        return self.net(x)

# AQUÍ CREAMOS EL MODELO
model = NeuralNARX(input_size=2 * LAGS)  # 2*6 = 12 entradas
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()


# ============================================================
# H) AQUÍ ENTRENAMOS EL MODELO
# ============================================================
print("\nEntrenando NARX...")

for epoch in range(1, EPOCHS + 1):
    model.train()

    # 1) Reiniciar gradientes
    optimizer.zero_grad()

    # 2) Predicción con datos train
    pred_train = model(X_train_t)

    # 3) Error (loss)
    loss_train = loss_fn(pred_train, y_train_t)

    # 4) Backprop
    loss_train.backward()

    # 5) Actualizar pesos
    optimizer.step()

    # AQUÍ SOLO MOSTRAMOS PROGRESO CADA 50 ÉPOCAS
    if epoch == 1 or epoch % 50 == 0:
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test_t)
            loss_test = loss_fn(pred_test, y_test_t).item()
        print(f"Epoch {epoch:3d} | Train Loss: {loss_train.item():.3f} | Test Loss: {loss_test:.3f}")


# ============================================================
# I) OPCIÓN 1: SELECCIONAR UN GRUPO (PROVEEDOR) Y PREDECIR
# ============================================================

def _split_groups(cell: str):
    """
    Esta función sirve para separar una celda si trae varios proveedores.
    Por ejemplo:
       "CON" -> ["CON"]
       "CON, BIM" -> ["CON","BIM"]
       "CON / BIM" -> ["CON","BIM"]
    """
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []
    s = str(cell).strip()
    if not s:
        return []
    parts = re.split(r"[,\;/\|\+]+", s)
    return [p.strip() for p in parts if p.strip()]

def get_groups(df: pd.DataFrame, group_col: str = GROUP_COL):
    """
    Esta función sirve para listar TODOS los grupos disponibles.
    Aquí es donde detectamos los proveedores del dataset.
    """
    groups = set()
    for v in df[group_col].values:
        for g in _split_groups(v):
            groups.add(g)
    return sorted(groups)

def predict_by_group(
    df: pd.DataFrame,
    model,
    lags: int = 6,
    group: str = "ALL",
    group_col: str = GROUP_COL,
    product_col: str = PRODUCT_COL,
    meta_cols=META_COLS,
    exo_mode: str = "global"    # global = total de todo el dataset
):
    """
    Esta función sirve para predecir:
       - group="ALL": predice todos los productos
       - group="CON": predice SOLO los productos del grupo CON

    exo_mode:
      - "global": usa exógena total semanal de TODO el dataset (recomendado)
      - "group" : usa exógena total semanal solo del grupo filtrado
    """

    # 1) Volvemos a construir la matriz (productos, semanas)
    week_cols_local = [c for c in df.columns if c not in meta_cols]
    weeks_df_all = df[week_cols_local].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    data_all = weeks_df_all.values.astype(np.float32)

    # 2) Filtrar filas por grupo
    if group is None or str(group).upper() == "ALL":
        df_group = df.copy()
        data_group = data_all
    else:
        group = str(group).strip()
        mask = df[group_col].apply(lambda x: group in _split_groups(x))
        df_group = df.loc[mask].copy()
        data_group = data_all[mask.values]

    # Si no hay productos en ese grupo
    if df_group.empty:
        return pd.DataFrame(columns=[product_col, group_col, "Pred_Sig_Semana"])

    # 3) Exógena (total semanal)
    if exo_mode == "global":
        exo_total_local = data_all.sum(axis=0).astype(np.float32)
    elif exo_mode == "group":
        exo_total_local = data_group.sum(axis=0).astype(np.float32)
    else:
        raise ValueError("exo_mode debe ser 'global' o 'group'")

    # 4) Predicción 1 paso adelante (próxima semana)
    products_local = df_group[product_col].astype(str).values
    groups_raw = df_group[group_col].astype(str).values

    last_x = exo_total_local[-lags:]  # últimos 6 del total semanal

    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(data_group.shape[0]):
            last_y = data_group[i, -lags:]  # últimos 6 del producto
            x_in = np.concatenate([last_y, last_x]).astype(np.float32)
            pred = model(torch.tensor(x_in).unsqueeze(0)).item()
            preds.append(pred)

    preds = np.array(preds)
    preds_round = np.clip(np.rint(preds), 0, None).astype(int)

    results = pd.DataFrame({
        product_col: products_local,
        group_col: groups_raw,
        "Pred_Sig_Semana": preds_round
    }).sort_values("Pred_Sig_Semana", ascending=False).reset_index(drop=True)

    return results


# ============================================================
# J) AQUÍ ES DONDE EL USUARIO ELIGE EL GRUPO (MENÚ)
# ============================================================
groups = get_groups(df, GROUP_COL)

print("\nGrupos disponibles (columna Provedores):")
print("0) ALL (todos)")
for i, g in enumerate(groups, 1):
    print(f"{i}) {g}")

sel = int(input("\nSelecciona un grupo (0 = ALL): "))

if sel == 0:
    chosen_group = "ALL"
else:
    chosen_group = groups[sel - 1]

# AQUÍ SE HACE LA PREDICCIÓN SOLO DEL GRUPO ELEGIDO
res = predict_by_group(df, model, lags=LAGS, group=chosen_group, exo_mode="global")

print(f"\nPredicción para grupo: {chosen_group} (Top 20)")
print(res.head(20).to_string(index=False))
print("\nTOTAL del grupo:", int(res["Pred_Sig_Semana"].sum()))


# ============================================================
# K) AQUÍ CALCULAMOS MÉTRICAS DEL MODELO (TEST) Y GRAFICAMOS
# ============================================================
with torch.no_grad():
    y_pred_test = model(X_test_t).squeeze().numpy()
y_true_test = y_test

# MAE / RMSE (errores)
mae  = mean_absolute_error(y_true_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))

# MAPE (ignorando ceros)
mask = (y_true_test != 0)
mape = (np.mean(np.abs((y_true_test[mask] - y_pred_test[mask]) / y_true_test[mask])) * 100
        if np.any(mask) else np.nan)

print("\nMÉTRICAS TEST (global)")
print(f"MAE : {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.2f}%" if not np.isnan(mape) else "MAPE: N/A")

# Gráfica real vs predicho
plt.figure(figsize=(10, 4))
plt.plot(y_true_test, label="Real")
plt.plot(y_pred_test, label="Predicho")
plt.title("NARX (LAGS=6) - Real vs Predicho (TEST) - Dataset V4")
plt.xlabel("Muestras (TEST)")
plt.ylabel("Valor")
plt.legend()
plt.tight_layout()
plt.show()
